In [1]:
from pathlib import Path

import torch
import pandas as pd

from PIL import Image
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification
)


# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = Path(
    "/Users/md.minhajulislampranto/Desktop/ML_Research_project/dataset"
)

GEMINI_DIR = BASE_DIR / "AI" / "gemini"
GPT_DIR = BASE_DIR / "AI" / "gpt"
QWEN_DIR = BASE_DIR / "AI" / "qwen"
GENUINE_DIR = BASE_DIR / "genuine"


# ============================================================
# 2. MODEL
# ============================================================

MODEL_NAME = "delpot/steganograph-ia-detector"

print("Loading model...")

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("Device:", device)
print("Model labels:", model.config.id2label)


# ============================================================
# 3. VALID IMAGE EXTENSIONS
# ============================================================

AI_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}

GENUINE_EXTENSIONS = {
    ".tif",
    ".tiff"
}


# ============================================================
# 4. GET IMAGES FROM FIRST N FOLDERS
# ============================================================

def collect_images_from_folders(
    root_dir,
    n_folders,
    extensions
):
    """
    Select the first n folders and collect ALL valid images
    inside those folders.
    """

    folders = sorted(
        [
            p for p in root_dir.iterdir()
            if p.is_dir()
        ]
    )[:n_folders]

    images = []

    print(f"\n{root_dir.name}")
    print("-" * 60)

    for folder in folders:

        folder_images = sorted(
            [
                p for p in folder.iterdir()
                if (
                    p.is_file()
                    and p.suffix.lower() in extensions
                )
            ]
        )

        print(
            f"{folder.name}: "
            f"{len(folder_images)} images"
        )

        images.extend(folder_images)

    return images


Loading model...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Device: cpu
Model labels: {0: 'real', 1: 'ai_generated'}


In [2]:
# ============================================================
# 5. GET FIRST N DIRECT IMAGES
# ============================================================

def collect_direct_images(
    root_dir,
    n_images,
    extensions
):
    """
    Select first n valid images directly inside root_dir.
    """

    files = sorted(
        [
            p for p in root_dir.iterdir()
            if (
                p.is_file()
                and p.suffix.lower() in extensions
            )
        ]
    )

    return files[:n_images]


# ============================================================
# 6. COLLECT DATASET
# ============================================================

print("\n" + "=" * 70)
print("COLLECTING DATASET")
print("=" * 70)


# Gemini:
# first 20 folders
# ALL images inside those folders
gemini_images = collect_images_from_folders(
    GEMINI_DIR,
    n_folders=20,
    extensions=AI_EXTENSIONS
)


# GPT:
# first 200 direct images
gpt_images = collect_direct_images(
    GPT_DIR,
    n_images=200,
    extensions=AI_EXTENSIONS
)


# Qwen:
# first 20 folders
# ALL images inside those folders
qwen_images = collect_images_from_folders(
    QWEN_DIR,
    n_folders=20,
    extensions=AI_EXTENSIONS
)


# Genuine:
# first 10 folders
# ALL .tif images inside those folders
genuine_images = collect_images_from_folders(
    GENUINE_DIR,
    n_folders=10,
    extensions=GENUINE_EXTENSIONS
)


# ============================================================
# 7. PRINT DATASET COUNTS
# ============================================================

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(f"Gemini  : {len(gemini_images)}")
print(f"GPT     : {len(gpt_images)}")
print(f"Qwen    : {len(qwen_images)}")
print(f"Genuine : {len(genuine_images)}")

total_images = (
    len(gemini_images)
    + len(gpt_images)
    + len(qwen_images)
    + len(genuine_images)
)

print("-" * 70)
print(f"TOTAL   : {total_images}")



COLLECTING DATASET

gemini
------------------------------------------------------------
1: 9 images
10: 9 images
11: 9 images
12: 9 images
13: 9 images
14: 9 images
15: 9 images
16: 9 images
17: 9 images
18: 9 images
19: 9 images
2: 9 images
20: 9 images
21: 9 images
22: 9 images
23: 9 images
24: 9 images
25: 9 images
26: 9 images
27: 9 images

qwen
------------------------------------------------------------
55: 12 images
56: 12 images
57: 12 images
58: 12 images
59: 12 images
60: 12 images
61: 12 images
62: 12 images
63: 12 images
64: 12 images
65: 12 images
66: 12 images
67: 12 images
68: 12 images
69: 12 images
70: 12 images
71: 12 images
72: 12 images
73: 12 images
74: 12 images

genuine
------------------------------------------------------------
1: 20 images
10: 10 images
100: 10 images
11: 10 images
12: 10 images
13: 10 images
14: 10 images
15: 10 images
16: 10 images
17: 10 images

DATASET SUMMARY
Gemini  : 180
GPT     : 198
Qwen    : 240
Genuine : 110
-----------------------

In [3]:
# ============================================================
# 8. BUILD DATASET
# ============================================================

dataset = []


for path in gemini_images:

    dataset.append({
        "path": path,
        "source": "Gemini",
        "true_label": "AI"
    })


for path in gpt_images:

    dataset.append({
        "path": path,
        "source": "GPT",
        "true_label": "AI"
    })


for path in qwen_images:

    dataset.append({
        "path": path,
        "source": "Qwen",
        "true_label": "AI"
    })


for path in genuine_images:

    dataset.append({
        "path": path,
        "source": "Genuine",
        "true_label": "Real"
    })


print(
    f"\nImages prepared for testing: {len(dataset)}"
)



Images prepared for testing: 728


In [4]:

# ============================================================
# 9. PREDICTION
# ============================================================

def predict_image(image_path):

    try:

        image = Image.open(
            image_path
        ).convert("RGB")

        inputs = processor(
            image,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():

            outputs = model(
                **inputs
            )

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )[0]

        predicted_id = torch.argmax(
            probabilities
        ).item()

        predicted_label = model.config.id2label[
            predicted_id
        ]

        confidence = probabilities[
            predicted_id
        ].item()


        # Model's labels:
        # real = 0
        # ai_generated = 1

        label = predicted_label.lower()

        if label in {
            "ai_generated",
            "ai",
            "artificial",
            "fake"
        }:

            final_label = "AI"

        elif label in {
            "real",
            "human"
        }:

            final_label = "Real"

        else:

            raise ValueError(
                f"Unknown model label: "
                f"{predicted_label}"
            )


        return final_label, confidence


    except Exception as e:

        print(
            f"\nError processing:\n"
            f"{image_path}\n"
            f"{e}"
        )

        return None, None

In [5]:
# ============================================================
# 10. RUN MODEL
# ============================================================

results = []

print("\n" + "=" * 70)
print("RUNNING AI IMAGE DETECTOR")
print("=" * 70)

for item in tqdm(
    dataset,
    desc="Processing images"
):

    prediction, confidence = predict_image(
        item["path"]
    )

    if prediction is None:
        continue

    results.append({

        "path": str(
            item["path"]
        ),

        "source": item["source"],

        "true_label": item["true_label"],

        "predicted_label": prediction,

        "confidence": confidence
    })



RUNNING AI IMAGE DETECTOR


Processing images: 100%|██████████| 728/728 [01:07<00:00, 10.75it/s]


In [6]:
# ============================================================
# 11. DATAFRAME
# ============================================================

df = pd.DataFrame(results)


print(
    f"\nSuccessfully processed: {len(df)} images"
)


# ============================================================
# 12. SAVE RAW RESULTS
# ============================================================

df.to_csv(
    "hf_detector_results.csv",
    index=False
)

print(
    "\nSaved:"
    "\nhf_detector_results.csv"
)



Successfully processed: 728 images

Saved:
hf_detector_results.csv


In [7]:
# ============================================================
# 13. OVERALL METRICS
# ============================================================

y_true = df["true_label"]
y_pred = df["predicted_label"]


accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    pos_label="AI",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label="AI",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label="AI",
    zero_division=0
)


summary = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Precision (AI)",
        "Recall (AI)",
        "F1-score (AI)"
    ],

    "Score": [
        accuracy,
        precision,
        recall,
        f1
    ]

})


print("\n" + "=" * 70)
print("OVERALL PERFORMANCE")
print("=" * 70)

print(
    summary.to_string(
        index=False,
        formatters={
            "Score": "{:.4f}".format
        }
    )
)



OVERALL PERFORMANCE
        Metric  Score
      Accuracy 0.3077
Precision (AI) 0.9831
   Recall (AI) 0.1877
 F1-score (AI) 0.3152


In [8]:
# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_true,
    y_pred,

    labels=[
        "Real",
        "AI"
    ]
)


cm_df = pd.DataFrame(

    cm,

    index=[
        "Actual Real",
        "Actual AI"
    ],

    columns=[
        "Predicted Real",
        "Predicted AI"
    ]
)


print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm_df)


CONFUSION MATRIX
             Predicted Real  Predicted AI
Actual Real             108             2
Actual AI               502           116


In [9]:
# ============================================================
# 15. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(

    classification_report(

        y_true,
        y_pred,

        labels=[
            "Real",
            "AI"
        ],

        zero_division=0
    )
)



CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Real       0.18      0.98      0.30       110
          AI       0.98      0.19      0.32       618

    accuracy                           0.31       728
   macro avg       0.58      0.58      0.31       728
weighted avg       0.86      0.31      0.31       728



In [10]:
# ============================================================
# 16. PERFORMANCE BY SOURCE
# ============================================================

print("\n" + "=" * 70)
print("PER-SOURCE PERFORMANCE")
print("=" * 70)


source_results = []


for source in [
    "GPT",
    "Gemini",
    "Qwen",
    "Genuine"
]:

    subset = df[
        df["source"] == source
    ]

    if len(subset) == 0:
        continue


    correct = (
        subset["true_label"]
        ==
        subset["predicted_label"]
    ).sum()


    accuracy_source = (
        correct / len(subset)
    )


    source_results.append({

        "Source": source,

        "Images": len(subset),

        "Correct": int(correct),

        "Incorrect": int(
            len(subset) - correct
        ),

        "Accuracy": accuracy_source
    })


source_df = pd.DataFrame(
    source_results
)


print(

    source_df.to_string(

        index=False,

        formatters={
            "Accuracy":
                "{:.4f}".format
        }
    )
)


PER-SOURCE PERFORMANCE
 Source  Images  Correct  Incorrect Accuracy
    GPT     198        0        198   0.0000
 Gemini     180       45        135   0.2500
   Qwen     240       71        169   0.2958
Genuine     110      108          2   0.9818


In [11]:
# ============================================================
# 17. AI DETECTION RATE BY GENERATOR
# ============================================================

print("\n" + "=" * 70)
print("AI DETECTION RATE BY GENERATOR")
print("=" * 70)


detection_results = []


for source in [
    "GPT",
    "Gemini",
    "Qwen"
]:

    subset = df[
        df["source"] == source
    ]


    total = len(subset)


    detected_as_ai = (

        subset["predicted_label"]
        == "AI"

    ).sum()


    missed = (
        total
        - detected_as_ai
    )


    detection_rate = (

        detected_as_ai / total
        if total > 0
        else 0
    )


    detection_results.append({

        "Generator": source,

        "Total": total,

        "Detected AI":
            int(detected_as_ai),

        "Missed":
            int(missed),

        "Detection Rate":
            detection_rate
    })


detection_df = pd.DataFrame(
    detection_results
)


print(

    detection_df.to_string(

        index=False,

        formatters={
            "Detection Rate":
                "{:.4f}".format
        }
    )
)



AI DETECTION RATE BY GENERATOR
Generator  Total  Detected AI  Missed Detection Rate
      GPT    198            0     198         0.0000
   Gemini    180           45     135         0.2500
     Qwen    240           71     169         0.2958


In [12]:
genuine = df[
    df["source"] == "Genuine"
]


total_genuine = len(genuine)


false_positives = (

    genuine["predicted_label"]
    == "AI"

).sum()


correct_real = (

    genuine["predicted_label"]
    == "Real"

).sum()


false_positive_rate = (

    false_positives
    / total_genuine

    if total_genuine > 0
    else 0
)


print("\n" + "=" * 70)
print("GENUINE IMAGE PERFORMANCE")
print("=" * 70)

print(
    f"Total genuine images : {total_genuine}"
)

print(
    f"Correctly Real       : {correct_real}"
)

print(
    f"False Positives      : {false_positives}"
)

print(
    f"False Positive Rate  : "
    f"{false_positive_rate:.4f}"
)




GENUINE IMAGE PERFORMANCE
Total genuine images : 110
Correctly Real       : 108
False Positives      : 2
False Positive Rate  : 0.0182


In [13]:
genuine = df[
    df["source"] == "Genuine"
]


total_genuine = len(genuine)


false_positives = (

    genuine["predicted_label"]
    == "AI"

).sum()


correct_real = (

    genuine["predicted_label"]
    == "Real"

).sum()


false_positive_rate = (

    false_positives
    / total_genuine

    if total_genuine > 0
    else 0
)


print("\n" + "=" * 70)
print("GENUINE IMAGE PERFORMANCE")
print("=" * 70)

print(
    f"Total genuine images : {total_genuine}"
)

print(
    f"Correctly Real       : {correct_real}"
)

print(
    f"False Positives      : {false_positives}"
)

print(
    f"False Positive Rate  : "
    f"{false_positive_rate:.4f}"
)




GENUINE IMAGE PERFORMANCE
Total genuine images : 110
Correctly Real       : 108
False Positives      : 2
False Positive Rate  : 0.0182


In [14]:
# ============================================================
# 19. FINAL SUMMARY TABLE
# ============================================================

final_summary = pd.DataFrame({

    "Metric": [

        "Total Images",

        "AI Images",

        "Real Images",

        "Accuracy",

        "Precision (AI)",

        "Recall (AI)",

        "F1-score (AI)"
    ],

    "Value": [

        len(df),

        len(
            df[df["true_label"] == "AI"]
        ),

        len(
            df[df["true_label"] == "Real"]
        ),

        accuracy,

        precision,

        recall,

        f1
    ]

})


print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(
    final_summary.to_string(
        index=False,
        formatters={
            "Value": lambda x:
                f"{x:.4f}"
                if isinstance(x, float)
                else str(x)
        }
    )
)


FINAL SUMMARY
        Metric    Value
  Total Images 728.0000
     AI Images 618.0000
   Real Images 110.0000
      Accuracy   0.3077
Precision (AI)   0.9831
   Recall (AI)   0.1877
 F1-score (AI)   0.3152


In [15]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
from tqdm import tqdm
import torch
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. LOAD SECOND MODEL
# ============================================================

MODEL_NAME_2 = "Smogy/SMOGY-Ai-images-detector"

print("Loading:", MODEL_NAME_2)

processor_2 = AutoImageProcessor.from_pretrained(
    MODEL_NAME_2
)

model_2 = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME_2
)

model_2.to(device)
model_2.eval()

print("Labels:", model_2.config.id2label)


# ============================================================
# 2. PREDICTION FUNCTION
# ============================================================

def predict_image_model_2(image_path):

    image = Image.open(
        image_path
    ).convert("RGB")

    inputs = processor_2(
        image,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model_2(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    predicted_id = torch.argmax(
        probabilities
    ).item()

    raw_label = model_2.config.id2label[
        predicted_id
    ]

    confidence = probabilities[
        predicted_id
    ].item()


    # Model labels:
    # artificial -> AI
    # human      -> Real

    if raw_label.lower() == "artificial":

        prediction = "AI"

    elif raw_label.lower() == "human":

        prediction = "Real"

    else:

        raise ValueError(
            f"Unknown label: {raw_label}"
        )


    return prediction, confidence

Loading: Smogy/SMOGY-Ai-images-detector


Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]

Labels: {0: 'artificial', 1: 'human'}


In [16]:

# ============================================================
# 3. RUN ON THE SAME DATASET
# ============================================================

results_2 = []

print("\nRunning second detector...\n")

for item in tqdm(
    dataset,
    desc="Second model"
):

    prediction, confidence = predict_image_model_2(
        item["path"]
    )

    results_2.append({

        "path": str(item["path"]),

        "source": item["source"],

        "true_label": item["true_label"],

        "predicted_label": prediction,

        "confidence": confidence
    })


df_2 = pd.DataFrame(results_2)


# ============================================================
# 4. METRICS
# ============================================================

y_true_2 = df_2["true_label"]
y_pred_2 = df_2["predicted_label"]


accuracy_2 = accuracy_score(
    y_true_2,
    y_pred_2
)

precision_2 = precision_score(
    y_true_2,
    y_pred_2,
    pos_label="AI",
    zero_division=0
)

recall_2 = recall_score(
    y_true_2,
    y_pred_2,
    pos_label="AI",
    zero_division=0
)

f1_2 = f1_score(
    y_true_2,
    y_pred_2,
    pos_label="AI",
    zero_division=0
)


# ============================================================
# 5. PRINT RESULT
# ============================================================

print("\n" + "=" * 60)
print("SECOND MODEL RESULT")
print("=" * 60)

print(
    f"Accuracy        : {accuracy_2:.4f}"
)

print(
    f"Precision (AI)  : {precision_2:.4f}"
)

print(
    f"Recall (AI)     : {recall_2:.4f}"
)

print(
    f"F1-score (AI)   : {f1_2:.4f}"
)


# ============================================================
# 6. CONFUSION MATRIX
# ============================================================

cm_2 = confusion_matrix(
    y_true_2,
    y_pred_2,
    labels=["Real", "AI"]
)

cm_2_df = pd.DataFrame(
    cm_2,
    index=[
        "Actual Real",
        "Actual AI"
    ],
    columns=[
        "Predicted Real",
        "Predicted AI"
    ]
)

print("\nConfusion Matrix:")
print(cm_2_df)


Running second detector...



Second model: 100%|██████████| 728/728 [01:33<00:00,  7.78it/s]


SECOND MODEL RESULT
Accuracy        : 0.4341
Precision (AI)  : 1.0000
Recall (AI)     : 0.3333
F1-score (AI)   : 0.5000

Confusion Matrix:
             Predicted Real  Predicted AI
Actual Real             110             0
Actual AI               412           206


In [17]:
comparison = pd.DataFrame({
    "Model": [
        "Steganograph-IA",
        "SMOGY"
    ],

    "Accuracy": [
        accuracy,
        accuracy_2
    ],

    "Precision": [
        precision,
        precision_2
    ],

    "Recall": [
        recall,
        recall_2
    ],

    "F1": [
        f1,
        f1_2
    ]
})

print(comparison.to_string(
    index=False,
    formatters={
        "Accuracy": "{:.4f}".format,
        "Precision": "{:.4f}".format,
        "Recall": "{:.4f}".format,
        "F1": "{:.4f}".format
    }
))

          Model Accuracy Precision Recall     F1
Steganograph-IA   0.3077    0.9831 0.1877 0.3152
          SMOGY   0.4341    1.0000 0.3333 0.5000


In [19]:
summary_by_source = []

for source in ["Gemini", "Qwen", "GPT", "Genuine"]:

    subset = df[df["source"] == source]

    true_label = subset["true_label"]
    predicted_label = subset["predicted_label"]

    correct = (
        true_label == predicted_label
    ).sum()

    total = len(subset)

    accuracy = correct / total

    if source == "Genuine":
        correctly_detected = (
            predicted_label == "Real"
        ).sum()
    else:
        correctly_detected = (
            predicted_label == "AI"
        ).sum()

    detection_rate = (
        correctly_detected / total
    )

    summary_by_source.append({

        "Source": source,

        "Total Images": total,

        "Correct": correct,

        "Incorrect": total - correct,

        "Accuracy": accuracy,

        "Correct Detection Rate": detection_rate
    })


source_summary = pd.DataFrame(
    summary_by_source
)

print(
    source_summary.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4f}".format,
            "Correct Detection Rate": "{:.4f}".format
        }
    )
)

 Source  Total Images  Correct  Incorrect Accuracy Correct Detection Rate
 Gemini           180       45        135   0.2500                 0.2500
   Qwen           240       71        169   0.2958                 0.2958
    GPT           198        0        198   0.0000                 0.0000
Genuine           110      108          2   0.9818                 0.9818


In [18]:
summary_by_source = []

for source in ["Gemini", "Qwen", "GPT", "Genuine"]:

    subset = df_2[df_2["source"] == source]

    true_label = subset["true_label"]
    predicted_label = subset["predicted_label"]

    correct = (
        true_label == predicted_label
    ).sum()

    total = len(subset)

    accuracy = correct / total

    if source == "Genuine":
        correctly_detected = (
            predicted_label == "Real"
        ).sum()
    else:
        correctly_detected = (
            predicted_label == "AI"
        ).sum()

    detection_rate = (
        correctly_detected / total
    )

    summary_by_source.append({

        "Source": source,

        "Total Images": total,

        "Correct": correct,

        "Incorrect": total - correct,

        "Accuracy": accuracy,

        "Correct Detection Rate": detection_rate
    })


source_summary = pd.DataFrame(
    summary_by_source
)

print(
    source_summary.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4f}".format,
            "Correct Detection Rate": "{:.4f}".format
        }
    )
)

 Source  Total Images  Correct  Incorrect Accuracy Correct Detection Rate
 Gemini           180        1        179   0.0056                 0.0056
   Qwen           240      205         35   0.8542                 0.8542
    GPT           198        0        198   0.0000                 0.0000
Genuine           110      110          0   1.0000                 1.0000
